In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel




In [2]:
df = pd.read_csv("../data/potential-talents - Aspiring human resources - seeking human resources.csv")
pd.set_option("display.max_colwidth", None)
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


In [3]:
df = df.drop("fit",axis=1)

In [4]:
df.head()

,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500+
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+


In [5]:
print(df["connection"].unique())

['85' '500+ ' '44' '1' '61' '2' '390' '57' '82' '5' '7' '16' '212' '409'
 '52' '455' '174' '268' '50' '4' '40' '18' '349' '155' '39' '64' '9' '415'
 '19' '71' '48' '103' '49']


In [6]:
df['connection'] = df['connection'].replace("500+ ","500")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          104 non-null    int64 
 1   job_title   104 non-null    object
 2   location    104 non-null    object
 3   connection  104 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.4+ KB


In [8]:
df = df.drop_duplicates(subset=['job_title'])

In [9]:
df.head()


,id,job_title,location,connection
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44
3,4,People Development Coordinator at Ryan,"Denton, Texas",500
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500


In [10]:
df.columns

Index(['id', 'job_title', 'location', 'connection'], dtype='object')

In [11]:
df[df["location"].str.contains(r"[^\x00-\x7F]", regex=True, na=False)]

,id,job_title,location,connection
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500
77,78,Human Resources Generalist at Schwan's,Amerika Birleşik Devletleri,500
93,94,Seeking Human Resources Opportunities. Open to travel and relocation.,Amerika Birleşik Devletleri,415


In [12]:
df['location'] = df['location'].str.replace("ü","u")
df['location'] = df['location'].str.replace("ş","s")
df['location'] = df['location'].str.replace("İ","I")

In [13]:
df["all_data"] = (
    df["job_title"].astype(str) + " " +
    df["location"].astype(str) + " " +
    df["connection"].astype(str)
)

In [58]:
df['all_data'][95]

'Student at Indiana University Kokomo - Business Management - \nRetail Manager at Delphi Hardware and Paint Lafayette, Indiana 19'

In [ ]:
labels_hr_resources = [1,0,1,0,0,1,1,1,1,0,0,
                       1,1,1,1,1,1,1,1,1,1,1,
                       1,1,1,1,1,1,0,1,1,1,1,
                       0,0,0,1,1,0,0,0,0,1,0,
                       0,0,1,1,1,0,0,0]

In [14]:
nlp = spacy.load("en_core_web_sm")

df['all_data'] = df['all_data'].astype("str").str.strip()

In [15]:
#!python -m spacy download en_core_web_sm

The idea is to build a function and apply it to each row in the job title column. The function tokenizes the text, removes punctuation and stop words, converts the words to lowercase, lemmatizes them, and returns the cleaned words as a list for each row in the job_title and return the join str becasue each model will use its own tokenization method.

In [16]:
def clean_title(x):
    doc = nlp(x)

    clean_text = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            clean_text.append(token.lemma_.lower())

    return " ".join(clean_text)

In [17]:
df["clean_job_title"] = df["all_data"].apply(clean_title)

In [18]:
df['clean_job_title']

0                            2019 c.t. bauer college business graduate magna cum laude aspire human resources professional houston texas 85
1                                                                              native english teacher epik english program korea kanada 500
2                                                               aspiring human resources professional raleigh durham north carolina area 44
3                                                                                      people development coordinator ryan denton texas 500
4                                                                            advisory board member celal bayar university izmir turkiye 500
5                                                                          aspiring human resources specialist greater new york city area 1
6                                                                      student humber college aspiring human resources generalist kanada 61
7                   

In [19]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

BoW  is defined as how many times does the word appear?
TF-IDF is defined as how important/distinctive is this word compared with the rest of the dataset?

In [20]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

In [21]:
score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [22]:
df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.4][["job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,job_title,similarity_1_bow
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.745356
99,Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis,0.615457
5,Aspiring Human Resources Specialist,0.577350
6,Student at Humber College and Aspiring Human Resources Generalist,0.577350
2,Aspiring Human Resources Professional,0.547723
71,Business Management Major and Aspiring Human Resources Manager,0.547723
26,Aspiring Human Resources Management student seeking an internship,0.522233
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.500000
27,Seeking Human Resources Opportunities,0.436436
87,Human Resources Management Major,0.436436


In [23]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.4][["job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,job_title,similarity_2_bow
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.730297
99,Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis,0.603023
87,Human Resources Management Major,0.534522
73,Human Resources Professional,0.534522
27,Seeking Human Resources Opportunities,0.534522
88,Director Human Resources at EY,0.500000
77,Human Resources Generalist at Schwan's,0.500000
66,"Human Resources, Staffing and Recruiting Professional",0.471405
5,Aspiring Human Resources Specialist,0.471405
98,Seeking Human Resources Position,0.471405


In [24]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

In [25]:
df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

,job_title,similarity_1_tfidf
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.547299
5,Aspiring Human Resources Specialist,0.416505
26,Aspiring Human Resources Management student seeking an internship,0.390410
2,Aspiring Human Resources Professional,0.365241
71,Business Management Major and Aspiring Human Resources Manager,0.361899
6,Student at Humber College and Aspiring Human Resources Generalist,0.345649
99,Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis,0.308499


In [26]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

,job_title,similarity_2_tfidf
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.491453


In [27]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')

Each word has a GloVe vector → average all word vectors → get one vector representing the entire job title.

In [28]:
# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)

In [29]:
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

In [30]:
X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)


In [31]:
df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

,job_title,similarity_1_glove
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.849718
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.821569
27,Seeking Human Resources Opportunities,0.811086
83,Human Resources professional for the world leader in GIS software,0.809051
88,Director Human Resources at EY,0.802835
26,Aspiring Human Resources Management student seeking an internship,0.800376
9,Seeking Human Resources HRIS and Generalist Positions,0.798086
73,Human Resources Professional,0.797593
5,Aspiring Human Resources Specialist,0.788184
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.784895


In [32]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)




,job_title,similarity_2_glove
27,Seeking Human Resources Opportunities,0.874183
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.871159
9,Seeking Human Resources HRIS and Generalist Positions,0.863006
93,Seeking Human Resources Opportunities. Open to travel and relocation.,0.862108
88,Director Human Resources at EY,0.826413
87,Human Resources Management Major,0.825759
73,Human Resources Professional,0.820881
83,Human Resources professional for the world leader in GIS software,0.809376
26,Aspiring Human Resources Management student seeking an internship,0.808565
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.801996


In [33]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)


,job_title,similarity_1_bert
27,Seeking Human Resources Opportunities,0.749111
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.747322
81,Aspiring Human Resources Professional | An energetic and Team-Focused Leader,0.735730
66,"Human Resources, Staffing and Recruiting Professional",0.720959
26,Aspiring Human Resources Management student seeking an internship,0.720783
65,Experienced Retail Manager and aspiring Human Resources Professional,0.719675
6,Student at Humber College and Aspiring Human Resources Generalist,0.719633
83,Human Resources professional for the world leader in GIS software,0.715453
103,Director Of Administration at Excellence Logging,0.707068
5,Aspiring Human Resources Specialist,0.702381


In [35]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

,job_title,similarity_2_bert
27,Seeking Human Resources Opportunities,0.801287
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.702566
66,"Human Resources, Staffing and Recruiting Professional",0.700950
9,Seeking Human Resources HRIS and Generalist Positions,0.698048
98,Seeking Human Resources Position,0.684404
91,Seeking employment opportunities within Customer Service or Patient Care,0.683962
103,Director Of Administration at Excellence Logging,0.683022
3,People Development Coordinator at Ryan,0.679889
73,Human Resources Professional,0.679089
88,Director Human Resources at EY,0.666376


In [36]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [37]:
s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.3][["all_data", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

,all_data,similarity_1_sbert
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.819170
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.788344
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.775312
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.731115
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.710243
81,"Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174",0.704329
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,0.702629
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.701894
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.699898
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.670924


In [38]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.3][["all_data", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

,all_data,similarity_2_sbert
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.761732
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.734525
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.717399
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.717316
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.699338
100,"Human Resources Generalist at Loparex Raleigh-Durham, North Carolina Area 500",0.671056
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.661074
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.658575
81,"Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174",0.653204
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.640741


In [39]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [40]:
X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)

In [41]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["all_data", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

,all_data,similarity_1_5e
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.900310
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.894750
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.890414
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.886353
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.878086
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.876692
81,"Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174",0.864373
71,"Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",0.863725
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.863497
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.860379


In [42]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["all_data", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

,all_data,similarity_2_5e
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.892398
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.890624
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.882211
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.876815
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.872307
93,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0.872267
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.865797
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.865030
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.860166
74,"Nortia Staffing is seeking Human Resources, Payroll & Administrative Professionals!! (408) 709-2621 San Jose, California 500",0.859875


In [43]:
df['all_data'][27]

'Seeking Human Resources Opportunities Chicago, Illinois 390'

In [44]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [45]:
X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

In [46]:
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["all_data", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

,all_data,similarity_1_bge
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.786999
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.783562
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.780927
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.778894
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.767548
99,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",0.755488
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.754207
71,"Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",0.747347
81,"Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174",0.743157
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,0.731464


In [47]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["all_data", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)

,all_data,similarity_2_bge
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.828740
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.774525
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.768068
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.761926
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.758372
93,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0.736612
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.730112
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.729630
66,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500",0.724346
74,"Nortia Staffing is seeking Human Resources, Payroll & Administrative Professionals!! (408) 709-2621 San Jose, California 500",0.721670


Among the evaluated embedding approaches, E5 produced the most relevant job-title rankings. This is consistent with E5's retrieval-oriented training, which is specifically designed to represent semantically related texts close together in the embedding space.

In [114]:
comparison_1 = pd.DataFrame({
    "BOW": df.nlargest(52,"similarity_1_bow")["all_data"].values,
    "TF-IDF": df.nlargest(52, "similarity_1_tfidf")["all_data"].values,
    "GloVe": df.nlargest(52, "similarity_1_glove")["all_data"].values,
    "BERT": df.nlargest(52, "similarity_1_bert")["all_data"].values,
    "SBERT": df.nlargest(52,"similarity_1_sbert")["all_data"].values,
    "E5": df.nlargest(52, "similarity_1_5e")["all_data"].values,
    "BGE": df.nlargest(52, "similarity_1_bge")["all_data"].values
})

comparison_1.index = range(1, 53)
comparison_1

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
3,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212"
4,Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Human Resources professional for the world leader in GIS software Highland, California 50","Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1
5,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Position Las Vegas, Nevada Area 48","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500"
6,"Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Experienced Retail Manager and aspiring Human Resources Professional Austin, Texas Area 57","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103"
7,"Aspiring Human Re

In [113]:
comparison_2 = pd.DataFrame({
    "BOW": df.nlargest(52,"similarity_2_bow")["all_data"].values,
    "TF-IDF": df.nlargest(52, "similarity_2_tfidf")["all_data"].values,
    "GloVe": df.nlargest(52, "similarity_2_glove")["all_data"].values,
    "BERT": df.nlargest(52, "similarity_2_bert")["all_data"].values,
    "SBERT": df.nlargest(52,"similarity_2_sbert")["all_data"].values,
    "E5": df.nlargest(52, "similarity_2_5e")["all_data"].values,
    "BGE": df.nlargest(52, "similarity_2_bge")["all_data"].values
})

comparison_2.index = range(1, 53)
pd.set_option("display.max_colwidth", None)
comparison_2

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Position Las Vegas, Nevada Area 48","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Human Resources Professional Greater Boston Area 16,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500
3,"Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
4,"Human Resources Management Major Milpitas, California 18",Aspiring Human Resources Specialist Greater New York City Area 1,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Opportunities Chicago, Illinois 390",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Position Las Vegas, Nevada Area 48"
5,Human Resources Professional Greater Boston Area 16,"Human Resources Management Major Milpitas, California 18",Director Human Resources at EY Greater Atlanta Area 349,"Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
6,Human Resources Generalist at Schwan's Amerika Birlesik Devletleri 500,Director Human Resources at EY Greater Atlanta Area 349,"Human Resources Management Major Milpitas, California 18","Seeking employment opportunities within Customer Service or Patient Care Torrance, California 64","Human Resources Generalist at Loparex Raleigh-Durham, North Carolina Area 500",Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415
7,Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",Human Resources Professional Greater Boston Area 16,"Director Of Administration at Excellence Logging Katy, Texas 500","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,Aspiring Human Resources Specialist Greater New York City Area 1
8,"Seeking Human

In [110]:
len(df['all_data'])

52

In [50]:
star_candidate = X_e5[6] # student humber college aspiring human resource

starred_similarity = cosine_similarity(
    X_e5,
    star_candidate.reshape(1, -1)
).flatten()

alpha = 0.6

df['rerank_score'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["all_data", "rerank_score"]].sort_values(
    "rerank_score",
    ascending=False
).head(10)

,all_data,rerank_score
27,"Seeking Human Resources Opportunities Chicago, Illinois 390",0.825121
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,0.801664
72,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0.799745
9,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0.799514
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0.790488
98,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0.785686
5,Aspiring Human Resources Specialist Greater New York City Area 1,0.773650
93,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0.771461
26,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",0.770525
75,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212",0.767285


In [51]:
star_candidates = X_e5[2:6] # student humber college aspiring human resource
star_anchor = star_candidates.mean(axis=0)
starred_similarity = cosine_similarity(
    X_e5,
    star_anchor.reshape(1, -1)
).flatten()

alpha = 0.5

df['rerank_scores'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["job_title", "rerank_scores"]].sort_values(
    "rerank_scores",
    ascending=False
).head(10)

,job_title,rerank_scores
27,Seeking Human Resources Opportunities,0.851140
2,Aspiring Human Resources Professional,0.848382
9,Seeking Human Resources HRIS and Generalist Positions,0.836446
72,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.829493
5,Aspiring Human Resources Specialist,0.828090
98,Seeking Human Resources Position,0.824765
75,Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment,0.811959
26,Aspiring Human Resources Management student seeking an internship,0.810844
93,Seeking Human Resources Opportunities. Open to travel and relocation.,0.808721
73,Human Resources Professional,0.804489


In [54]:

from transformers import pipeline

zero_shot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

text = "is this candidate HR related, yes or no."

candidate_labels = [
    "Yes",
    "No",
]

result = zero_shot_classifier(
    text,
    candidate_labels=candidate_labels
)

print(result)


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

c:\Users\msafw\anaconda3\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\msafw\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'is this candidate HR related, yes or no.', 'labels': ['Yes', 'No'], 'scores': [0.5841009616851807, 0.41589903831481934]}


In [63]:
def zero_shot_classifer_check():
    zero_shot_classifier = pipeline(
        task="zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    candidate_labels = [
        "This candidate HR related",
        "This candidate not HR related"
    ]

    texts = df["all_data"].tolist()

    results = zero_shot_classifier(
        texts,
        candidate_labels=candidate_labels
    )

    return results

In [64]:
results = zero_shot_classifer_check()
df_results = pd.DataFrame(results)
df_results

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

,sequence,labels,scores
0,"2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional Houston, Texas 85","[This candidate HR related, This candidate not HR related]","[0.9924140572547913, 0.007585996761918068]"
1,Native English Teacher at EPIK (English Program in Korea) Kanada 500,"[This candidate not HR related, This candidate HR related]","[0.5356791615486145, 0.4643208086490631]"
2,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","[This candidate HR related, This candidate not HR related]","[0.9939702749252319, 0.0060297283343970776]"
3,"People Development Coordinator at Ryan Denton, Texas 500","[This candidate HR related, This candidate not HR related]","[0.9079433679580688, 0.09205660969018936]"
4,"Advisory Board Member at Celal Bayar University Izmir, Turkiye 500","[This candidate HR related, This candidate not HR related]","[0.7797142863273621, 0.22028574347496033]"
5,Aspiring Human Resources Specialist Greater New York City Area 1,"[This candidate HR related, This candidate not HR related]","[0.9948657155036926, 0.005134335719048977]"
6,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"[This candidate HR related, This candidate not HR related]","[0.9923202991485596, 0.0076797460205852985]"
7,HR Senior Specialist San Francisco Bay Area 500,"[This candidate HR related, This candidate not HR related]","[0.9229919910430908, 0.0770079717040062]"
8,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"[This candidate HR related, This candidate not HR related]","[0.9908297061920166, 0.009170301258563995]"
9,"Student at Chapman University Lake Forest, California 2","[This candidate not HR related, This candidate HR related]","[0.6715776920318604, 0.32842233777046204]"


In [65]:
comparison_1

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44"
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7"
3,Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212"
4,Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Human Resources professional for the world leader in GIS software Highland, California 50","Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",Aspiring Human Resources Specialist Greater New York City Area 1
5,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Seeking Human Resources Position Las Vegas, Nevada Area 48","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500"
6,"Business Management Major and Aspiring Human Resources Manager Monroe, Louisiana Area 5",Student at Humber College and Aspiring Human Resources Generalist Kanada 61,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500","Experienced Retail Manager and aspiring Human Resources Professional Austin, Texas Area 57","Aspiring Human Resources Professional | An energetic and Team-Focused Leader Austin, Texas Area 174","Aspiring Human Resources Professional | Passionate about helping to create an inclusive and engaging work environment New York, New York 212","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103"
7,"Aspiring Human Re

In [117]:
labels_aspiring = [1,0,1,0,0,1,1,0,
                   0,0,0,0,1,0,1,0,0,0,0,1,0,0,
                   0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,
                   0,0,0,0,0,0,0,0,0,1,0,0,0,0,0]
labels_seeking = [0,0,0,0,0,0,0,1,
                  1,0,0,0,0,1,0,0,0,1,0,0,1,0
                  ,1,0,0,0,0,0,0,0,0,0,0,0,0,0,
                  0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0]

In [118]:
comparison_1["relevance"] = labels_aspiring
comparison_2["relevance"] = labels_seeking

In [119]:
comparison_2

,BOW,TF-IDF,GloVe,BERT,SBERT,E5,BGE,relevance
1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Position Las Vegas, Nevada Area 48","Seeking Human Resources Opportunities Chicago, Illinois 390","Seeking Human Resources Opportunities Chicago, Illinois 390",0
2,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Human Resources Professional Greater Boston Area 16,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44","Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,0
3,"Seeking Human Resources Opportunities Chicago, Illinois 390","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Human Resources, Staffing and Recruiting Professional Jackson, Mississippi Area 500",Aspiring Human Resources Specialist Greater New York City Area 1,"Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7","Aspiring Human Resources Manager, seeking internship in Human Resources. Houston, Texas Area 7",0
4,"Human Resources Management Major Milpitas, California 18",Aspiring Human Resources Specialist Greater New York City Area 1,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Opportunities Chicago, Illinois 390",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Seeking Human Resources Position Las Vegas, Nevada Area 48",0
5,Human Resources Professional Greater Boston Area 16,"Human Resources Management Major Milpitas, California 18",Director Human Resources at EY Greater Atlanta Area 349,"Seeking Human Resources Position Las Vegas, Nevada Area 48",Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area 500,"Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103","Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area 44",0
6,Human Resources Generalist at Schwan's Amerika Birlesik Devletleri 500,Director Human Resources at EY Greater Atlanta Area 349,"Human Resources Management Major Milpitas, California 18","Seeking employment opportunities within Customer Service or Patient Care Torrance, California 64","Human Resources Generalist at Loparex Raleigh-Durham, North Carolina Area 500",Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,Seeking Human Resources Opportunities. Open to travel and relocation. Amerika Birlesik Devletleri 415,0
7,Director Human Resources at EY Greater Atlanta Area 349,"Aspiring Human Resources Management student seeking an internship Houston, Texas Area 500",Human Resources Professional Greater Boston Area 16,"Director Of Administration at Excellence Logging Katy, Texas 500","Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis Cape Girardeau, Missouri 103",Aspiring Human Resources Specialist Greater New York City Area 1,Aspiring Human Resources Specialist Greater New York City A